# robo-pi obstacle detector — custom YOLOv8s training

Run this notebook in **Google Colab** with a GPU runtime (Runtime → Change runtime type → T4 GPU).

**Before running:** fill in the three `PARAM` variables in Cell 1 with your Roboflow credentials.

In [ ]:
# ── Cell 1 · Parameters + setup ──────────────────────────────────────────────
# Fill these in before running. Get them from roboflow.com → Settings → API.
ROBOFLOW_API_KEY  = "YOUR_API_KEY"      # roboflow.com → Settings → Roboflow API
ROBOFLOW_WORKSPACE = "YOUR_WORKSPACE"   # your workspace slug (shown in the URL)
ROBOFLOW_PROJECT   = "YOUR_PROJECT"     # your project slug
ROBOFLOW_VERSION   = 1                  # dataset version number to download

from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics roboflow -q
import ultralytics
ultralytics.checks()

In [ ]:
# ── Cell 2 · Download dataset from Roboflow ───────────────────────────────────
# Exports in YOLOv8 format: images + labels + data.yaml
from roboflow import Roboflow

rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset = project.version(ROBOFLOW_VERSION).download("yolov8")

print(f"Dataset location: {dataset.location}")
print(f"data.yaml:        {dataset.location}/data.yaml")

In [ ]:
# ── Cell 3 · Train ────────────────────────────────────────────────────────────
# yolov8s.pt — small model, significantly better than nano for custom classes.
# imgsz=640 matches the robot's lores stream resolution.
# epochs=50 is safe for 300-500 images; increase to 100 if you have 1000+.
!yolo train \
    model=yolov8s.pt \
    data={dataset.location}/data.yaml \
    imgsz=640 \
    epochs=50 \
    batch=16 \
    name=robo_pi_obstacle \
    exist_ok=True

In [ ]:
# ── Cell 4 · Validate ─────────────────────────────────────────────────────────
# Check mAP50, precision, and recall on the validation split.
# Aim for mAP50 > 0.7 before deploying. If lower, add more varied training images.
!yolo val \
    model=runs/detect/robo_pi_obstacle/weights/best.pt \
    data={dataset.location}/data.yaml \
    imgsz=640

In [ ]:
# ── Cell 5 · Export to ONNX ───────────────────────────────────────────────────
# simplify=True reduces the graph for faster OpenCV DNN inference on the Pi.
!yolo export \
    model=runs/detect/robo_pi_obstacle/weights/best.pt \
    format=onnx \
    imgsz=640 \
    simplify=True

import os
onnx_path = "runs/detect/robo_pi_obstacle/weights/best.onnx"
size_mb   = os.path.getsize(onnx_path) / 1e6
print(f"Exported: {onnx_path}  ({size_mb:.1f} MB)")

In [ ]:
# ── Cell 6 · Save to Google Drive ─────────────────────────────────────────────
# After this cell completes, download best.onnx from Google Drive to your Mac,
# then scp it to the Pi: scp best.onnx pi@<pi-ip>:~/robo-pi/src/ai/models/
import shutil, pathlib

dest = pathlib.Path("/content/drive/MyDrive/robo_pi_best.onnx")
shutil.copy(onnx_path, dest)
print(f"Saved to Google Drive: {dest}")
print()
print("Next steps on the Pi:")
print("  1. scp ~/Downloads/robo_pi_best.onnx pi@<pi-ip>:~/robo-pi/src/ai/models/best.onnx")
print("  2. In config/hardware.yaml, update obstacle_avoidance:")
print("       yolo_model:       'src/ai/models/best.onnx'")
print("       yolo_input_size:  640")
print("       yolo_class_names: ['obstacle']")
print("  3. python3 -m src.perception.vision.object_detection  # verify on Pi")